# Scraping de coches de segunda mano 

### Que queremos 
- hacer scraping de paginas web de coches de segunda mano
- 
- utilizar rag para guardar los precios de los coches


### login en el modelo 

In [1]:
import os
from openai import OpenAI
from dotenv import load_dotenv

load_dotenv()
api_key = os.getenv('GOOGLE_API_KEY')

MODEL = "gemini-2.0-flash"
openai = OpenAI(base_url="https://generativelanguage.googleapis.com/v1beta", api_key=api_key)

response = openai.chat.completions.create(
 model=MODEL,
 messages=[{"role": "user", "content": "¿Cuánto son 2 + 2?"}]
)

print(response.choices[0].message.content)

2 + 2 = 4



### Diseño final del scraping 

In [3]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

def scrape_autofesa_final():
    """Scraper mejorado para extraer todos los coches y precios aunque cambien las clases"""
    options = Options()
    # options.add_argument('--headless')  # Temporalmente desactivado para asegurar carga
    options.add_argument('--window-size=1920,1080')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36')
    
    driver = webdriver.Chrome(options=options)
    resultados = []
    
    try:
        url = "https://www.autofesa.com/coches-segunda-mano"
        print(f"🔗 Accediendo a: {url}")
        driver.get(url)
        
        wait = WebDriverWait(driver, 15)
        wait.until(EC.presence_of_element_located((By.CSS_SELECTOR, ".vehicle-list__item")))
        time.sleep(2)

        # ==== SCROLL INFINITO ====
        SCROLL_PAUSE_TIME = 2
        last_height = driver.execute_script("return document.body.scrollHeight")

        while True:
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            time.sleep(SCROLL_PAUSE_TIME)
            new_height = driver.execute_script("return document.body.scrollHeight")
            if new_height == last_height:
                break
            last_height = new_height

        car_elements = driver.find_elements(By.CSS_SELECTOR, ".vehicle-list__item")
        print(f"🚗 Encontrados {len(car_elements)} coches después del scroll")
        
        for idx, car in enumerate(car_elements, 1):
            try:
                # Título
                try:
                    title_elem = car.find_element(By.CSS_SELECTOR, ".vehicle-card__title")
                    title = title_elem.text.strip()
                except:
                    title = "Sin título"
                
                # Precio (varios selectores posibles)
                price = "Precio no disponible"
                price_selectors = [
                    ".vehicle-card__price",  # selector normal
                    ".vehicle-card__price--soldout",  # coches sin stock o vendidos
                    ".vehicle-card__price span"  # fallback a span
                ]
                for sel in price_selectors:
                    try:
                        price_elem = car.find_element(By.CSS_SELECTOR, sel)
                        if price_elem.text.strip():
                            price = price_elem.text.strip()
                            break
                    except:
                        continue
                
                # Link
                try:
                    link_elem = car.find_element(By.CSS_SELECTOR, "a")
                    link = link_elem.get_attribute("href")
                except:
                    link = "Sin enlace"
                
                # Información adicional
                try:
                    features = car.find_elements(By.CSS_SELECTOR, ".vehicle-card__features .list .item")
                    additional_info = " | ".join([f.text.strip() for f in features if f.text.strip()])
                except:
                    additional_info = "Sin información adicional"
                
                resultados.append({
                    "#": idx,
                    "Modelo": title,
                    "Precio": price,
                    "Información": additional_info,
                    "Link": link
                })
                
                if idx <= 5:
                    print(f"   ✓ Coche {idx}: {title[:30]}... - {price}")
                    
            except Exception as e:
                print(f"   ❌ Error procesando coche {idx}: {e}")
                continue
    
    except Exception as e:
        print(f"❌ Error general: {e}")
    
    finally:
        driver.quit()
    
    return resultados

# Ejecutar scraper
print("🚀 Iniciando scraper mejorado...")
resultados_final = scrape_autofesa_final()
print(f"\n✅ Scraping completado. Total: {len(resultados_final)} coches extraídos.")


There was an error managing chromedriver (error sending request for url (https://googlechromelabs.github.io/chrome-for-testing/known-good-versions-with-downloads.json)); using driver found in the cache


🚀 Iniciando scraper mejorado...
🔗 Accediendo a: https://www.autofesa.com/coches-segunda-mano
🚗 Encontrados 30 coches después del scroll
   ✓ Coche 1: Abarth 500 1.4T JET 595 165CV ... - 17.350
€
OFERTA
   ✓ Coche 2: Aixam S10 SPORT S10 SPORT... - Precio no disponible
   ✓ Coche 3: Aixam S8 COUPE S8 COUPE 9CV AU... - 10.450
€
OFERTA
   ✓ Coche 4: Alfa Romeo Giulietta GIULIETTA... - 16.850
€
OFERTA
   ✓ Coche 5: Alfa Romeo Giulietta GIULIETTA... - 8.750
€
OFERTA

✅ Scraping completado. Total: 30 coches extraídos.


In [4]:
import pandas as pd

# Convertir lista de diccionarios a DataFrame
df = pd.DataFrame(resultados_final)

# Mostrar toda la tabla en consola
pd.set_option('display.max_rows', None)   # Mostrar todas las filas
pd.set_option('display.max_columns', None)  # Mostrar todas las columnas
pd.set_option('display.width', 1000)  # Evitar que se corte horizontalmente

print(df)


     #                                             Modelo                       Precio                                        Información                                               Link
0    1  Abarth 500 1.4T JET 595 165CV 70TH ANNIVERSARY...            17.350\n€\nOFERTA  2020 | 83.900km | Gasolina | Manual | Utilitar...  https://www.autofesa.com/coches-de-ocasion/aba...
1    2                          Aixam S10 SPORT S10 SPORT         Precio no disponible  2024 | 3.274km | Diesel | Manual | Utilitario ...  https://www.autofesa.com/coches-de-ocasion/aix...
2    3        Aixam S8 COUPE S8 COUPE 9CV AUTO 3P # CUERO            10.450\n€\nOFERTA  2015 | 44.144km | Diesel | Automático | Utilit...  https://www.autofesa.com/coches-de-ocasion/aix...
3    4  Alfa Romeo Giulietta GIULIETTA 1.4T 170CV LUSS...            16.850\n€\nOFERTA  2018 | 39.200km | Gasolina | Automático | Util...  https://www.autofesa.com/coches-de-ocasion/alf...
4    5  Alfa Romeo Giulietta GIULIETTA 1.6 JTD DISTINT.